In [1]:
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

In [2]:
PROJECT_DIR = Path.cwd().resolve().parents[1] / 'anaconda'

RAW_HTML_DIR = PROJECT_DIR / 'data' / 'raw' / 'html'

print(PROJECT_DIR)
print(RAW_HTML_DIR)

D:\anaconda
D:\anaconda\data\raw\html


In [29]:
SITE_BASE_URLS = {
    1: 'https://www.dalunacats.com',
    2: 'https://dogmaru.co.kr',
    3: 'https://nebalhouse.com',
    4: 'https://doremicat.co.kr',
}

DETAIL_KEY_MAP = {
    '묘종': 'breed_detail',
    '성별': 'gender',
    '나이': 'age',
    '모색': 'color',
    '특징': 'feature',
    '지점': 'branch',
}

In [4]:
batch_dirs = sorted(
    [
        path
        for path in RAW_HTML_DIR.iterdir()
        if path.is_dir()
    ]
)

batch_dirs

[WindowsPath('D:/anaconda/data/raw/html/20260827_232246'),
 WindowsPath('D:/anaconda/data/raw/html/20260827_233751')]

In [5]:
BATCH_DIR = batch_dirs[-1]

print(BATCH_DIR)

D:\anaconda\data\raw\html\20260827_233751


In [6]:
html_files = sorted(
    BATCH_DIR.glob('site_*.html')
)

print(f'HTML 파일 수 : {len(html_files)}')

html_files[:5]

HTML 파일 수 : 1601


[WindowsPath('D:/anaconda/data/raw/html/20260827_233751/site_01_detail_0001.html'),
 WindowsPath('D:/anaconda/data/raw/html/20260827_233751/site_01_detail_0002.html'),
 WindowsPath('D:/anaconda/data/raw/html/20260827_233751/site_01_detail_0003.html'),
 WindowsPath('D:/anaconda/data/raw/html/20260827_233751/site_01_detail_0004.html'),
 WindowsPath('D:/anaconda/data/raw/html/20260827_233751/site_01_detail_0005.html')]

In [7]:
site_01_file = BATCH_DIR / 'site_01_page_001.html'

html = site_01_file.read_text(
    encoding='utf-8'
)

soup = BeautifulSoup(
    html,
    'html.parser'
)

print(type(soup))

<class 'bs4.BeautifulSoup'>


In [8]:
cat_cards = soup.select(
    'ul.cat_list li'
)

print(
    f'1번 사이트 1페이지 '
    f'고양이 카드 수 : {len(cat_cards)}'
)

1번 사이트 1페이지 고양이 카드 수 : 20


In [9]:
first_card = cat_cards[0]

# 1번 사이트 고양이 수집

In [21]:
def parse_site_01_card(card):
    link_tag = card.select_one('a')
    img_tag = card.select_one('.cat_img img')
    text_tags = card.select('.cat_txt p')

    breed = text_tags[0].get_text(strip=True)
    name = text_tags[1].get_text(strip=True)

    detail_path = link_tag.get('href')
    img_path = img_tag.get('src')

    base_url = SITE_BASE_URLS[1]

    detail_url = urljoin(
        base_url,
        detail_path,
    )

    img_url = urljoin(
        base_url,
        img_path,
    )

    return {
        'name': name,
        'breed': breed,
        'detail_url': detail_url,
        'img_url': img_url,
    }

In [27]:
def parse_site_01_detail(file_path):
    html = file_path.read_text(
        encoding='utf-8'
    )

    soup = BeautifulSoup(
        html,
        'html.parser'
    )

    spec_items = soup.select(
        'ul.cat_spec li'
    )

    detail_data = {}

    for item in spec_items:
        key_tag = item.select_one('span')
        value_tag = item.select_one('p')

        if key_tag is None or value_tag is None:
            continue

        key = key_tag.get_text(strip=True)
        value = value_tag.get_text(strip=True)

        column_name = DETAIL_KEY_MAP.get(key)

        if column_name is not None:
            detail_data[column_name] = value

    return detail_data

In [25]:
site_01_files = sorted(
    BATCH_DIR.glob('site_01_page_*.html')
)

sample_detail = parse_site_01_detail(
    site_01_detail_files[0]
)

sample_detail

site_01_cats = []

for file_path in site_01_files:
    html = file_path.read_text(
        encoding='utf-8'
    )

    soup = BeautifulSoup(
        html,
        'html.parser'
    )

    cat_cards = soup.select(
        'ul.cat_list li'
    )

    for card in cat_cards:
        cat = parse_site_01_card(card)
        site_01_cats.append(cat)

In [17]:
print(
    f'목록 HTML 수 : {len(site_01_files)}'
)

print(
    f'상세 HTML 수 : {len(site_01_detail_files)}'
)

목록 HTML 수 : 74
상세 HTML 수 : 1462


In [18]:
site_01_df = pd.DataFrame(site_01_cats)

site_01_df

,name,breed,detail_url,img_url
0,홀앙,브리티쉬 먼치킨,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
1,쨈,브리티쉬 먼치킨,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
2,리즈,랙돌,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
3,우디,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
4,통깨,노르웨이숲,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
...,...,...,...,...
1457,레이,브리티쉬 먼치킨,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
1458,테이,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
1459,릿지,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
1460,볼링이,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...


In [30]:
sample_detail = parse_site_01_detail(
    site_01_detail_files[0]
)

sample_detail

{'breed_detail': '브리티쉬 먼치킨',
 'gender': '여아',
 'age': '2개월령',
 'color': '골드바이',
 'feature': '왕눈이에 생긴 것 마냥 성격도 순둥이!',
 'branch': '왕십리점'}

In [32]:
site_01_detail_data = []

for index, file_path in enumerate(
    site_01_detail_files,
    start=1,
):
    detail_data = parse_site_01_detail(
        file_path
    )

    site_01_detail_data.append(
        detail_data
    )

    if index % 100 == 0:
        print(
            f'{index} / '
            f'{len(site_01_detail_files)} '
            f'파싱 완료'
        )

100 / 1462 파싱 완료
200 / 1462 파싱 완료
300 / 1462 파싱 완료
400 / 1462 파싱 완료
500 / 1462 파싱 완료
600 / 1462 파싱 완료
700 / 1462 파싱 완료
800 / 1462 파싱 완료
900 / 1462 파싱 완료
1000 / 1462 파싱 완료
1100 / 1462 파싱 완료
1200 / 1462 파싱 완료
1300 / 1462 파싱 완료
1400 / 1462 파싱 완료


In [33]:
site_01_detail_df = pd.DataFrame(
    site_01_detail_data
)

site_01_detail_df

,breed_detail,gender,age,color,feature,branch
0,브리티쉬 먼치킨,여아,2개월령,골드바이,왕눈이에 생긴 것 마냥 성격도 순둥이!,왕십리점
1,브리티쉬 먼치킨,남아,2개월령,크림바이,오밀조밀 공주 이목구비지만 멋쟁이 왕자다냥!,잠실점
2,랙돌,여아,2개월령,블루바이,품에 안기는 순간 인형처럼 스르륵 녹아내리는 원조 봉제인형 냥이,잠실점
3,브리티쉬숏헤어,남아,2개월령,블루골드,느긋한 발걸음으로 집사 뒤를 묵묵히 쫓아오는 든든한 친구,왕십리점
4,노르웨이숲,여아,2개월령,블루태비앤화이트,,왕십리점
...,...,...,...,...,...,...
1457,브리티쉬 먼치킨,남아,2개월령,크림바이,사랑스러운 장난꾸러기 ♥,
1458,브리티쉬숏헤어,남아,2개월령,크림태비,차분하고 조용한 사랑둥이 ♥,
1459,브리티쉬숏헤어,남아,2개월령,화이트,호기심 많은 장난꾸러기 애교냥 ♥,
1460,브리티쉬숏헤어,여아,2개월령,블루바이,장난감을 좋아하는 쾌활한 아꺵이 ♥,


In [35]:
site_01_merged_df = pd.concat(
    [
        site_01_df.reset_index(drop=True),
        site_01_detail_df.reset_index(drop=True),
    ],
    axis=1,
)

site_01_merged_df

,name,breed,detail_url,img_url,breed_detail,gender,age,color,feature,branch
0,홀앙,브리티쉬 먼치킨,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,브리티쉬 먼치킨,여아,2개월령,골드바이,왕눈이에 생긴 것 마냥 성격도 순둥이!,왕십리점
1,쨈,브리티쉬 먼치킨,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,브리티쉬 먼치킨,남아,2개월령,크림바이,오밀조밀 공주 이목구비지만 멋쟁이 왕자다냥!,잠실점
2,리즈,랙돌,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,랙돌,여아,2개월령,블루바이,품에 안기는 순간 인형처럼 스르륵 녹아내리는 원조 봉제인형 냥이,잠실점
3,우디,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,브리티쉬숏헤어,남아,2개월령,블루골드,느긋한 발걸음으로 집사 뒤를 묵묵히 쫓아오는 든든한 친구,왕십리점
4,통깨,노르웨이숲,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,노르웨이숲,여아,2개월령,블루태비앤화이트,,왕십리점
...,...,...,...,...,...,...,...,...,...,...
1457,레이,브리티쉬 먼치킨,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,브리티쉬 먼치킨,남아,2개월령,크림바이,사랑스러운 장난꾸러기 ♥,
1458,테이,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,브리티쉬숏헤어,남아,2개월령,크림태비,차분하고 조용한 사랑둥이 ♥,
1459,릿지,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,브리티쉬숏헤어,남아,2개월령,화이트,호기심 많은 장난꾸러기 애교냥 ♥,
1460,볼링이,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,브리티쉬숏헤어,여아,2개월령,블루바이,장난감을 좋아하는 쾌활한 아꺵이 ♥,


In [38]:
site_01_merged_df[
    site_01_merged_df['breed']
    != site_01_merged_df['breed_detail']
][
    ['name', 'breed', 'breed_detail']
]

,name,breed,breed_detail


In [39]:
site_01_final_df = site_01_merged_df.drop(
    columns=['breed_detail']
)

site_01_final_df

,name,breed,detail_url,img_url,gender,age,color,feature,branch
0,홀앙,브리티쉬 먼치킨,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,여아,2개월령,골드바이,왕눈이에 생긴 것 마냥 성격도 순둥이!,왕십리점
1,쨈,브리티쉬 먼치킨,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,남아,2개월령,크림바이,오밀조밀 공주 이목구비지만 멋쟁이 왕자다냥!,잠실점
2,리즈,랙돌,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,여아,2개월령,블루바이,품에 안기는 순간 인형처럼 스르륵 녹아내리는 원조 봉제인형 냥이,잠실점
3,우디,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,남아,2개월령,블루골드,느긋한 발걸음으로 집사 뒤를 묵묵히 쫓아오는 든든한 친구,왕십리점
4,통깨,노르웨이숲,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,여아,2개월령,블루태비앤화이트,,왕십리점
...,...,...,...,...,...,...,...,...,...
1457,레이,브리티쉬 먼치킨,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,남아,2개월령,크림바이,사랑스러운 장난꾸러기 ♥,
1458,테이,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,남아,2개월령,크림태비,차분하고 조용한 사랑둥이 ♥,
1459,릿지,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,남아,2개월령,화이트,호기심 많은 장난꾸러기 애교냥 ♥,
1460,볼링이,브리티쉬숏헤어,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...,여아,2개월령,블루바이,장난감을 좋아하는 쾌활한 아꺵이 ♥,


In [40]:
site_01_final_df.isnull().sum()

name          0
breed         0
detail_url    0
img_url       0
gender        0
age           0
color         0
feature       0
branch        0
dtype: int64

In [41]:
print(
    'detail_url 중복 수 :',
    site_01_final_df['detail_url'].duplicated().sum()
)

print(
    '전체 행 중복 수 :',
    site_01_final_df.duplicated().sum()
)

detail_url 중복 수 : 0
전체 행 중복 수 : 0


In [42]:
site_01_final_df['breed'].value_counts()

breed
브리티쉬숏헤어        397
브리티쉬 먼치킨       182
랙돌              93
데본렉스            83
스코티쉬폴드          64
아메리칸숏헤어         55
러시안블루           54
페르시안            49
먼치킨 미뉴엣         47
노르웨이숲           46
샴               43
아비시니안           41
메인쿤             33
뱅갈              31
스핑크스            21
셀커크렉스           20
아메리칸컬           18
먼치킨 래가퍼         18
브리티쉬롱헤어         17
먼치킨             17
먼치킨 킬트          11
싱가푸라            10
노르웨이 숲          10
하이랜드폴드          10
먼치킨 킨카로우        10
먼치킨 램킨           9
터키쉬앙고라           9
노르웨이숲 먼치킨        6
먼치킨 밤비노          6
엑죠틱              5
렉돌               5
네바마스커레이드         4
페르시안 익스트림        4
페르시안 친칠라         4
아메리칸 컬           3
벵갈               3
브리티쉬먼치킨          2
먼치킨 드월프          2
네벨룽              2
스코티쉬스트레이트        2
발리니즈             1
제네타 먼치킨          1
스코티쉬 스트레이트       1
킨카로우             1
브리티쉿쇼헤어          1
먼치               1
먼치킨 민스킨          1
셀커크랙스            1
먼치킨 미뉴엣 (롱)      1
브리티쉬 먼처킨         1
샴 먼치킨            1
액죠틱 먼치킨          1
터키시앙고라

In [44]:
from datetime import datetime

today = datetime.now().strftime('%Y%m%d')

PARSED_DIR = PROJECT_DIR / 'data' / 'processed'

PARSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PARSED_CSV_PATH = (
    PARSED_DIR
    / f'site_01_parsed_cats_{today}.csv'
)

site_01_final_df.to_csv(
    PARSED_CSV_PATH,
    index=False,
    encoding='utf-8-sig',
)

print(PARSED_CSV_PATH)

D:\anaconda\data\processed\site_01_parsed_cats_20260828.csv
